# Lab · A team of agents: specialists, a reviewer, a supervisor

**AI Agents course · Day 4 (multi-agent systems and human-in-the-loop)**

Yesterday one agent held every tool. Today the same job is done by a **team**: a triage agent that decides who is needed, a policy specialist and an account specialist that each know one thing well, a writer, a **reviewer who is not the writer**, and a supervisor that fixes the order, enforces a budget and knows when to hand a case to a person.

```
message ─▶ triage ─▶ which specialists? ─▶ policy specialist (Day 2: retrieval)    ─┐
                    │                       account specialist (SQL homework)       ├─▶ case file ─▶ writer ─▶ reviewer ──returned──▶ writer again, or a specialist again
                    └─ escalate now ─▶ a person                                     ┘                                └──approved──▶ the reply      ──refused twice──▶ a person
```

Then the honest question: **is the team better than one agent with all the tools?** The same three cases go through both, judged by the same reviewer, and a table shows what each path cost and what it got right. That trade-off comes back in the Day 5 capstone, where you build an orchestrator with workers.

**Your part:** four short exercises, all of them text, no code: the **triage prompt** (Exercise 1), the **writer's rules** (Exercise 2), a **tone-only reviewer** (Exercise 3) and a **customer message** of your own (Exercise 4). Everything else is given code: read it, run it, and read the trace.

### Setup: Runtime → Change runtime type → **T4 GPU**, then run this cell

In [ ]:
!pip install -q -U transformers accelerate chromadb sentence-transformers pypdf

import torch, json, re

In [ ]:
#@title Load Qwen3.5-4B (the same model in every lab)
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText

MODEL_ID = "Qwen/Qwen3.5-4B"                     # about 9 GB in float16: fits a free Colab T4
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.float16 if torch.cuda.is_available() else torch.float32, device_map="auto")
model.eval()
print("running on:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
#@title chat(): the one function that talks to the model (the same in every lab)
def chat(messages, system=None, max_new_tokens=400, temperature=0.0, thinking=False, tools=None):
    """A prompt string or a list of {"role", "content"} messages -> the model's reply as text.
    system: an optional system prompt, put in front of the messages.
    temperature=0 is greedy (the same reply every time); above 0 it samples, with top_p=0.8 and top_k=20
    from the Qwen3.5 model card. thinking=True lets the model reason in <think>...</think> before answering.
    tools: tool schemas for native tool calling (the reply then may contain <tool_call> blocks)."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if system:
        messages = [{"role": "system", "content": system}] + messages
    prompt = tok.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True,
                                     enable_thinking=thinking)                 # wrap in the model's own chat markup
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    sampling = dict(do_sample=True, temperature=temperature, top_p=0.8, top_k=20) if temperature > 0 else dict(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **sampling)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()  # keep only the new text

In [ ]:
print(chat("Say hello in five words."))

## 1: The cases and the case file

Five customer messages from the Brightwave story. The **case file** is a plain dictionary: every agent reads it and writes its own part. That one rule is what makes a team readable afterward: the file *is* the conversation, and the trace says who touched it and what it cost. The case file also adds up its own cost (model calls and seconds), which the comparison at the end needs.

In [ ]:
import time, sqlite3, pandas as pd

MESSAGES = {
"amina": """From: Amina Hassan
Subject: Credit balance refund

My account has been in credit for four months and I would like the money back rather than leaving it on the account.
I pay by direct debit and I sent a meter reading on 2 September.""",
"tom": """From: Tom Okafor
Subject: Missed smart meter appointment

Nobody came for the smart meter installation on Tuesday and no one called. I took the whole morning off work for it.
Please tell me what compensation I am owed and book a new slot.""",
"bayo": """From: Bayo Adeyemi
Subject: Complaint about waiting times

I have called four times this month and waited over forty minutes each time before giving up.
I want the goodwill payment you promised in July and never paid, and I want this treated as a formal complaint.""",
"sofia": """From: Sofia Pereira
Subject: Refund after moving out

I moved out of the flat on 14 August and gave a closing reading the same day. The final statement shows I am owed 62 pounds.
Three weeks later there is still no refund in my bank account.""",
"thanks": """From: Grace Osei
Subject: Thank you

Just a quick note to say the engineer yesterday was brilliant. Sorted the meter in twenty minutes. Thank you!""",
}

def new_case(message):
    """The shared case file. Every agent reads it and writes its own part; agents never talk to each other directly."""
    return {"message": message, "customer": re.search(r"^From:\s*(.+)$", message, re.M).group(1).strip(),
            "kind": None, "urgency": None, "needs": [], "escalate": False, "why": "", "findings": {}, "draft": None, "review": None,
            "rounds": 0, "calls": 0, "seconds": 0.0}

TRACE = []

def record(case, agent, note, started, calls):
    """Print what an agent did, and keep it: this trace is how the run is read afterward. The case file adds up its own cost."""
    seconds = time.time() - started
    case["calls"] += calls
    case["seconds"] += seconds
    TRACE.append({"customer": case["customer"], "agent": agent, "model calls": calls, "seconds": round(seconds, 1)})
    print(f"\n{'━' * 72}\n {agent.upper():<10} {case['customer']}  ·  {calls} call{'s' if calls != 1 else ''}, {seconds:.0f}s\n{'━' * 72}\n{note}")

print(MESSAGES["tom"])

## 2: What the specialists know

Six short Brightwave policies, indexed for search the way the Day 2 RAG Pipeline lab does it (MiniLM embeddings in ChromaDB), and a ten-customer table in SQLite for the account specialist to query, as in the SQL homework. Two specialists, two knowledge sources, and neither can see the other's. Read the `smart_meter` and `escalation` policies and the `balance_gbp`, `contacts_this_month` and `notes` columns: the cases below turn on them.

In [ ]:
# Six short internal documents of "Brightwave Energy", a fictional UK supplier with a Dublin office.
DOCS = [
 {"id": "refund_policy", "region": "UK", "text": """Credit balance refunds: customers who pay by direct debit can request a refund of their credit balance
at any time. Approved refunds are paid within 10 working days to the original payment method. Refunds above GBP 250 must first be
reviewed by the Billing team, who confirm an up-to-date meter reading. Duplicate payments are refunded in full within 5 working days.
Customers on the Priority Services Register are fast-tracked and paid within 5 working days."""},
 {"id": "complaints", "region": "UK", "text": """Complaints: we acknowledge every complaint within 2 working days and aim to resolve it within
10 working days. If a complaint is still open after 8 weeks, the customer may go to the Energy Ombudsman and we must tell them so in
writing. Frontline staff may offer goodwill payments up to GBP 50; larger amounts need a team leader."""},
 {"id": "smart_meter", "region": "UK", "text": """Smart meter installation: appointments are free, take 60-90 minutes and use 4-hour windows,
Monday to Saturday. Cancelling or moving an appointment is free up to 48 hours before. If our engineer misses an appointment without one
working day's notice, the customer receives GBP 30 compensation automatically within 10 working days."""},
 {"id": "leave_uk", "region": "UK", "text": """Annual leave, United Kingdom: full-time employees receive 25 days plus bank holidays. The leave year
runs January to December. Up to 5 unused days may be carried over into the next year and must be used by 31 March or they are lost."""},
 {"id": "leave_ireland", "region": "IE", "text": """Annual leave, Ireland: full-time employees in the Dublin office receive 22 days plus public
holidays. The leave year runs January to December. Up to 8 unused days may be carried over into the next year and must be used by 30 April."""},
 {"id": "escalation", "region": "ALL", "text": """Escalation rules for assistants: any mention of a gas smell or a safety risk -> give the National
Gas Emergency number 0800 111 999 and hand over to a human immediately. Refunds above GBP 250 -> hand over to Billing. Complaints open
for more than 8 weeks, legal threats, or a third contact about the same issue -> hand over to the Complaints team. Never disclose bank
details, passwords or another person's data."""},
]


import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

embedder = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2", device="cuda")
policies = chromadb.Client().create_collection("policies", embedding_function=embedder, metadata={"hnsw:space": "cosine"})
policies.add(ids=[d["id"] for d in DOCS], documents=[d["text"] for d in DOCS], metadatas=[{"source": d["id"]} for d in DOCS])

CUSTOMERS = pd.DataFrame([
 ("BW-1001", "Amina Hassan",     "London",     "Fixed 12",  86.40, 4, 0, 0, 1, "2025-09-02", "pays by direct debit"),
 ("BW-1002", "Tom Okafor",       "Midlands",   "Variable", -12.10, 0, 0, 0, 1, "2025-08-28", "smart meter install booked 9 Sep: engineer did not attend, no notice given"),
 ("BW-1003", "Erik Lindqvist",   "Scotland",   "Variable", -151.00, 0, 0, 0, 1, "2025-07-30", "Aug bill 74.00, Sep bill 151.00; Sep used an estimated reading"),
 ("BW-1004", "Sofia Pereira",    "London",     "Fixed 12",  61.50, 1, 0, 0, 2, "2025-08-14", "moved out 14 Aug; final statement issued 20 Aug; refund not yet paid"),
 ("BW-1005", "Claire Donnelly",  "North West", "Green Plus", -40.20, 0, 0, 0, 1, "2025-09-04", "smart meter fitted 1 Sep"),
 ("BW-1006", "Hannah Whitfield", "North West", "Fixed 24",  -33.00, 0, 0, 0, 1, "2025-09-01", "fixed tariff ends 2026-03-31"),
 ("BW-1007", "Bayo Adeyemi",     "Midlands",   "Variable",  -58.75, 0, 0, 1, 4, "2025-08-20", "goodwill payment of 40.00 promised 15 Jul, not paid"),
 ("BW-1008", "Petra Novak",      "Scotland",   "Fixed 12",  -21.00, 0, 0, 0, 2, "2025-08-30", "third-floor flat; two install visits declined by engineer"),
 ("BW-1009", "Lewis Campbell",   "Scotland",   "Variable",  12.00, 1, 1, 0, 0, "2025-09-01", "on the Priority Services Register"),
 ("BW-1010", "Grace Osei",       "London",     "Green Plus", -18.40, 0, 0, 0, 1, "2025-09-10", "smart meter fitted 10 Sep"),
], columns=["customer_id", "name", "region", "tariff", "balance_gbp", "months_in_credit", "priority_services", "open_complaints", "contacts_this_month", "last_reading", "notes"])

conn = sqlite3.connect(":memory:")
CUSTOMERS.to_sql("customers", conn, index=False)
SCHEMA = "customers(" + ", ".join(CUSTOMERS.columns) + ")  -- balance_gbp > 0 means the customer is in credit"
print(len(DOCS), "policies indexed ·", len(CUSTOMERS), "customers in SQLite")
CUSTOMERS[["customer_id", "name", "balance_gbp", "contacts_this_month", "notes"]].head(4)

## Step 1: Triage — the router

The first decision on every case: what kind of case, how urgent, **which specialists it needs**, and whether a person must take it now. The escalation rules come straight from the policy, so the model is applying them, not inventing them. A message that needs nobody should cost nothing more than this one call: the supervisor in Step 5 closes such a case right after triage.

Two of these decisions matter too much to leave to a 4B model alone, so `triage()` backs them with **rules in code** that run after the model and can only add caution:
- **a third contact goes to a person.** The escalation policy sends a third contact about the same issue to the Complaints team. The customer table has `contacts_this_month`, so code checks it: three or more and the case is escalated. (The table counts all contacts, not contacts about one issue, so this rule errs on the side of a person.)
- **anything that is not `other` gets the policy specialist**: a reply that promises money or action must be checked against the rules.

The cell below runs triage with a first prompt on all five messages and compares what **the model alone** decided, and what it became **with the rules**, with the routing we expect.

In [ ]:
#@title chat_json(): chat() that returns a dict
import json

def chat_json(messages, **kw):
    """chat(), but returns the first JSON object in the reply as a dict ({} if there is none).
    Small models sometimes write broken JSON: on a parse error, show the model the error and ask once more."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    for attempt in range(2):
        text = chat(messages, **kw)
        start = text.find("{")
        if start < 0:
            return {}
        try:
            return json.JSONDecoder().raw_decode(text[start:])[0]             # parse one object, ignore what follows
        except json.JSONDecodeError as err:
            messages = messages + [{"role": "assistant", "content": text},
                                   {"role": "user", "content": f"That JSON is invalid ({err}). Reply with only the corrected JSON."}]
    return {}

In [ ]:
TRIAGE_PROMPT = """You triage customer messages for an energy supplier. Two specialists exist: 'policy' knows what our rules allow, 'account' can look up the customer's record. Only list a specialist if this message needs it; a thank-you needs neither.

Escalate straight to a person only for: a gas smell or safety risk, a legal threat, or a customer whose message says this is their third or later contact about the same issue. Everything else, refunds and complaints included, stays with the team.

Message:
{message}"""

TRIAGE_JSON = ('\n\nReturn JSON: {"kind": "refund | appointment | billing | complaint | other", "urgency": "low | normal | high", '
               '"needs": [...], "escalate": true/false, "why": "one sentence"}')

def triage(case):
    """Read the message; decide what kind of case it is, who is needed, and whether a person must take it now."""
    started = time.time()
    reply = chat_json(TRIAGE_PROMPT.replace("{message}", case["message"]) + TRIAGE_JSON, max_new_tokens=200)
    case.update(kind=reply.get("kind"), urgency=reply.get("urgency"), escalate=str(reply.get("escalate")).lower() == "true",
                needs=[n for n in reply.get("needs") or [] if n in ("policy", "account")], why=str(reply.get("why", "")))
    case["model_route"] = {"needs": list(case["needs"]), "escalate": case["escalate"]}      # what the model alone decided
    # the rules in code: they run after the model and can only add caution
    contacts = conn.execute("SELECT contacts_this_month FROM customers WHERE name = ?", (case["customer"],)).fetchone()
    if contacts and contacts[0] >= 3 and not case["escalate"]:
        case["escalate"], case["why"] = True, f"RULE: the account shows {contacts[0]} contacts this month (policy: a third contact goes to the Complaints team)"
    if case["kind"] != "other" and "policy" not in case["needs"]:
        case["needs"].append("policy")
        case["why"] += "  [RULE: added 'policy']"
    record(case, "triage", f"{case['kind']} · {case['urgency']} · needs {case['needs'] or 'nobody'}"
           + ("  →  ESCALATE: " if case["escalate"] else "  ·  ") + case["why"], started, calls=1)
    return case

EXPECTED = {"amina": ["account", "policy"], "tom": ["account", "policy"], "bayo": "a person", "sofia": ["account", "policy"], "thanks": []}

def routing_check():
    """Triage all five messages and compare the model alone, and the model with the rules, with the routing we expect."""
    route = lambda d: "a person" if d["escalate"] else sorted(d["needs"])
    rows = []
    for key, expected in EXPECTED.items():
        case = triage(new_case(MESSAGES[key]))
        rows.append({"customer": case["customer"], "expected": expected, "model alone": route(case["model_route"]), "with the rules": route(case)})
    table = pd.DataFrame(rows)
    for col in ["model alone", "with the rules"]:
        print(f"{col + ':':<16} {sum(r == e for r, e in zip(table[col], table['expected']))} of {len(table)} routed as expected")
    return table

routing_check()

**What to look for.** In our test run the first prompt sent Amina, Tom and Sofia to the account specialist only, and Bayo too, without escalating him, although his message says he called four times: the model alone routed 1 of 5 as expected (Grace's thank-you). With the rules, all 5: the policy rule added the policy specialist to the four cases that are not `other`, and the contacts rule escalated Bayo from his account data. A writer that never gets a policy finding has no policy to quote; Step 3 shows what it writes instead.

### Exercise 1 · A triage prompt that routes

The rules are a safety net for two decisions. Everything else is still the model's call: the kind, the urgency, a gas smell, a legal threat, and every message no rule foresaw. Rewrite `TRIAGE_PROMPT` so that **the model alone** routes all five messages as expected: say what each specialist knows and when it is needed (most requests need both), and spell out the escalation conditions, including how to count the contacts a customer mentions. Keep `{message}` where the message goes; the code adds the JSON format.

In [ ]:
TRIAGE_PROMPT = """You triage customer messages for Brightwave Energy, an energy supplier. Two specialists can help:
- "policy" knows our rules: refunds, compensation, complaints, goodwill payments, appointments. List it whenever the customer asks for money, a refund, compensation, a booking or a complaint.
- "account" looks up this customer's record: balance, payments, notes on past visits. List it whenever the answer depends on this customer's own account.
Most requests need both. A thank-you that asks for nothing needs neither: "needs": [].

Set "escalate": true (a person takes the case now) if the message mentions any of:
- a gas smell or a safety risk;
- a legal threat;
- contacting us three or more times about this issue (count the calls, emails or visits the customer mentions).
Otherwise "escalate": false.

Message:
{message}"""

routing_check()

**What to look for.** The first line is the one to improve. In our test run the solution prompt routed all 5 as expected by itself and escalated Bayo for his four calls, so the rules had nothing left to add. The first prompt's "only list a specialist if this message needs it" had made the model list as few as possible; saying what each specialist is for, and that most requests need both, fixed that. Your prompt now drives every triage call in the rest of the lab; if your model-alone count is below 5, the rules are what keep the cases below on track. Five messages are a smoke test, not an evaluation: a sixth message could break your prompt.

## Step 2: The specialists

Each specialist has one job, one knowledge source and one short prompt. The **policy specialist** retrieves the two most relevant passages and states what they allow, with the source in brackets after every fact. The **account specialist** writes one SQL query, runs it, and repairs it once if the database objects: the SQL homework's repair loop in a few lines. Both write their findings into the case file and nothing else.

Notice what they do not see. The policy specialist never sees the account. The account specialist never sees the policy. Small prompts, no cross-contamination, and a finding that can be traced to its source. Steps 2 to 4 follow one case, Tom's.

In [ ]:
def policy_specialist(case, question=None):
    """What do our rules allow for this case? Two passages in, a short cited finding out."""
    started = time.time()
    r = policies.query(query_texts=[question or case["message"]], n_results=2)
    passages = "\n\n".join(f"[{m['source']}] {d}" for d, m in zip(r["documents"][0], r["metadatas"][0]))
    finding = chat(f"Policy passages:\n{passages}\n\nCustomer message:\n{case['message']}\n\n"
                   + (f"Specific question from the reviewer: {question}\n\n" if question else "")
                   + f"In 2-4 short sentences, state exactly what the policy allows or requires for this case, quoting every amount and timeframe, "
                   f"with the source id in square brackets after each fact. If the passages do not cover it, say so plainly.", max_new_tokens=220)
    case["findings"]["policy"] = finding
    record(case, "policy", finding, started, calls=1)
    return case

def account_specialist(case, question=None):
    """What does the customer's record show? One SQL query, repaired once if it fails."""
    started, calls, sql, error = time.time(), 0, None, None
    ask = question or "everything on this customer's record that matters for their message: " + case["message"].splitlines()[1]
    for attempt in range(2):
        reply = chat(f"Write one SQLite SELECT over the table {SCHEMA}.\nThe customer is {case['customer']!r}.\nQuestion: {ask}\n"
                     + (f"\nYour previous query:\n{sql}\nfailed with: {error}\nWrite a corrected query.\n" if error else "")
                     + "Reply with the SQL in a ```sql block.", max_new_tokens=150); calls += 1
        block = re.search(r"```sql\s*(.*?)```", reply, re.S | re.I)
        sql = (block.group(1) if block else reply).strip().rstrip(";")
        try:
            rows = pd.read_sql(sql, conn); break
        except Exception as e:
            error, rows = str(e).splitlines()[-1], None
    finding = rows.to_string(index=False) if rows is not None and len(rows) else f"could not read the account ({error or 'no matching record'})"
    case["findings"]["account"] = finding
    record(case, "account", f"{sql}\n\n{finding}", started, calls=calls)
    return case

SPECIALISTS = {"policy": policy_specialist, "account": account_specialist}

case = triage(new_case(MESSAGES["tom"]))
for name in case["needs"]:
    case = SPECIALISTS[name](case)

## Step 3: The writer

The writer sees the message and the specialists' findings, and is told to use **only** facts, amounts and timeframes from those findings. Its rules live in `WRITER_RULES`, so they can change without touching the function. It will not always obey. That is what the next agent is for, and what Exercise 2 is about. The helper `money()` lists every amount of money in a text; the reviewer uses it too.

In [ ]:
WRITER_RULES = """- Use only facts, amounts and timeframes that appear in the specialists' findings, and nothing from memory.
- Acknowledge the issue, say exactly what happens next and by when.
- Under 120 words; plain and warm; sign off as 'Brightwave Customer Team'."""

def money(text):
    """Every amount of money in a text, as a number: '£30', 'GBP 30' and '30 pounds' all give '30'."""
    found = re.findall(r"(?:£|GBP)\s?(\d[\d,]*(?:\.\d+)?)|(\d[\d,]*(?:\.\d+)?)\s*(?:pounds|GBP)\b", text)
    return {f"{float((a or b).replace(',', '')):g}" for a, b in found}

def writer(case, reasons=None):
    """Draft the reply from the case file. On a second round it also sees why the reviewer returned the last one."""
    started = time.time()
    findings = "\n\n".join(f"{name.upper()} SPECIALIST FOUND:\n{text}" for name, text in case["findings"].items()) or "No specialist findings."
    draft = chat(f"Write a reply to this customer on behalf of Brightwave Energy.\n\nCustomer message:\n{case['message']}\n\n{findings}\n\n"
                 f"Rules:\n{WRITER_RULES}"
                 + (f"\n\nThe reviewer returned your previous draft:\n{case['draft']}\n\nbecause:\n" + "\n".join("- " + r for r in reasons)
                    + "\n\nFix every point." if reasons else ""), max_new_tokens=260)
    case["draft"] = draft
    record(case, "writer", draft, started, calls=1)
    return case

case = writer(case)
print("\namounts in the draft:", money(case["draft"]) or "none")

# the same case file without the policy finding: what the writer gets when triage sends Tom to the account specialist only
no_policy = writer({**case, "findings": {k: v for k, v in case["findings"].items() if k != "policy"}})
print("\namounts in the draft:", money(no_policy["draft"]) or "none")

**What to look for.** In our test run, with both findings, the draft quoted the £30 from the policy finding but announced "we have booked an appointment for you for Monday, 9 September": nobody booked anything, and 9 September is the date of the visit that was missed. Without the policy finding, the draft called Tom's balance "£12.10 in compensation" (a balance of −12.10 is money Tom *owes*) and booked him for 28 August 2025, the date of his last meter reading. Both amounts appear in a finding, so the number check in Step 4 cannot object to either. Routing decides what the writer can know; its rules decide what it does with it.

### Exercise 2 · The writer's rules

Rewrite `WRITER_RULES`: keep the useful rules and add rules for what went wrong above. Three that matter in this lab:
- **a balance is not compensation**: `balance_gbp` above 0 is money we owe the customer, below 0 is money the customer owes us; a compensation amount may come only from the policy finding, and with no policy finding the reply names no amount;
- **nothing is "done" unless a finding says so**: no slot "has been booked", no payment "has been sent"; what the findings do not cover, a colleague follows up;
- one of the first rules asks for "exactly what happens next and by when", which invites a deadline even when no finding gives one.

The cell writes the same two drafts again with your rules: Tom's full case file, and the one without the policy finding.

In [ ]:
WRITER_RULES = """- Use only facts, amounts and timeframes that appear in the specialists' findings, and nothing from memory.
- A balance is not compensation. balance_gbp in the account record is the account balance (above 0: we owe the customer; below 0: the customer owes us); mention it only if the customer asks about their balance or a refund of it. Name a compensation amount only if the POLICY finding states it; if no policy finding states one, name no amount and say a colleague will confirm what is owed.
- Never say something has already been done (booked, paid, refunded, passed on) unless a finding says so. For a request the findings do not cover, such as booking a new appointment, write that a colleague will follow it up, without promising a date.
- Acknowledge the issue and say what happens next; give a timeframe only if a finding states it.
- Under 120 words; plain and warm; sign off as 'Brightwave Customer Team'."""

case = writer(case)
print("\namounts in the draft:", money(case["draft"]) or "none")

# the same case file without the policy finding: what the writer gets when triage sends Tom to the account specialist only
no_policy = writer({**case, "findings": {k: v for k, v in case["findings"].items() if k != "policy"}})
print("\namounts in the draft:", money(no_policy["draft"]) or "none")

**What to look for.** In our test run the solution rules fixed both drafts. With the full case file: £30, and "our team will follow up with you to book a suitable slot" instead of a booking. Without the policy finding: no amount at all, and "a colleague will confirm what you are owed". That second draft still says "I will also arrange a new appointment" before handing the booking to a colleague, a softer form of the same invention: rules make a mistake rarer, not impossible. The `amounts` line under each draft is the quickest check of your own rules.

## Step 4: The reviewer — a critic who is not the author

Half the checklist is code, half is the model, same split as every lab so far. Code checks that **every amount and timeframe in the draft appears somewhere in a specialist's finding**, that the draft avoids a short list of risky promises ("immediately", "guarantee", ...), and that the reply is not too long. Be clear about what that number check can and cannot do: it catches an amount nobody found, but it cannot tell *which* finding a number came from, so a balance quoted as compensation passes it. The model checks whether the draft actually answers what the customer asked, in the right tone, without claims the findings do not support. It is also told what the team cannot do (book appointments, make payments), so "a colleague will follow it up" counts as an answer rather than a gap. If a fact is missing, the reviewer names the specialist who should be asked again and the question to ask.

The verdict goes into the case file. The reviewer does not fix anything; that is the writer's job.

In [ ]:
BANNED = ["guarantee", "immediately", "straight away", "as a gesture we will", "we will definitely"]

def reviewer(case):
    """Approve the draft, or return it with reasons. Code checks the numbers, the model checks the sense."""
    started = time.time()
    draft, evidence = case["draft"], " ".join(case["findings"].values())
    known = {f"{float(n.replace(',', '')):g}" for n in re.findall(r"\d[\d,]*(?:\.\d+)?", evidence)}         # every number a specialist reported
    problems = [f"£{n} is not in any specialist's findings" for n in sorted(money(draft) - known)]          # money the draft states
    for span, n, unit in re.findall(r"\b((\d+)\s*(?:working\s+)?(day|hour|week|month)s?)\b", draft):            # timeframes the draft states
        if f"{float(n):g}" not in known or unit not in evidence.lower():
            problems.append(f"'{span}' is not in any specialist's findings")
    problems += [f"promises '{p}'" for p in BANNED if p in draft.lower()]
    if len(draft.split()) > 150:
        problems.append(f"{len(draft.split())} words, the limit is 150")
    verdict = chat_json(f"You review replies before they reach customers. A reply may use only what the specialists found. The team cannot book "
                        f"appointments, make payments or look anything else up: for such requests, saying that a colleague will follow it up is the "
                        f"correct answer, not a claim that something has been done.\n\nCustomer message:\n{case['message']}\n\nWhat the specialists found:\n{evidence}\n\n"
                        f"Draft reply:\n{draft}\n\nDoes the draft answer what the customer actually asked, in a plain and warm tone, without stating "
                        f"anything the findings do not support? Give at most two short reasons. If a needed fact is missing, name the specialist to ask ('policy' or 'account') and the question.\n"
                        f'Return JSON: {{"approved": true/false, "reasons": ["each problem, if any"], "ask_again": [], "question": ""}}', max_new_tokens=400)
    model_ok = str(verdict.get("approved")).lower() == "true"
    said = verdict.get("reasons") or []
    said = [said] if isinstance(said, str) else said
    approved = not problems and model_ok
    reasons = problems + ([] if model_ok else [str(r) for r in said if r])                # the model's reasons only when it objected
    case["review"] = {"approved": approved, "reasons": reasons,
                      "question": str(verdict.get("question") or "") or "; ".join(reasons[:2]),     # never send a specialist back with an empty question
                      "ask_again": [a for a in (verdict.get("ask_again") or []) if a in SPECIALISTS]}
    record(case, "reviewer", "APPROVED" if approved else "RETURNED\n" + "\n".join("- " + r for r in reasons)
           + (f"\n→ ask {case['review']['ask_again']}: {case['review']['question']}" if case["review"]["ask_again"] else ""), started, calls=1)
    return case

case = reviewer(case)

**What to look for.** In our test run the reviewer returned Tom's draft from Exercise 2. The code checks found nothing: £30 and 10 working days are both in a finding. The model objected that "our team will follow up" should say a colleague will follow up, and added a nitpick about the start time of the 4-hour windows, which no finding could supply. The model left the question empty, so the question is the reviewer's first two reasons, and it names the account specialist as the one to ask. A strict critic costs calls; Step 5 shows what the supervisor does with a returned draft.

## Step 5: The supervisor — order, budget, exit

Everything above is a function of the case file. The supervisor decides the order and nothing else: triage, then the specialists triage asked for, then writer and reviewer for at most two rounds. Three things make it an agent system rather than a pipeline: the reviewer can **send a specialist back to work** with a specific question, there is an **exit to a person**, taken when triage says so or when the reviewer refuses twice, and a message that needs nobody is **closed right after triage**. The person receives the whole case file, not a summary.

In [ ]:
RESULTS = {}                                                                       # customer -> how the team's run ended, and what it cost

def finish(case, outcome, results=None):
    """Write down how a case ended. The case file has counted its own model calls and seconds."""
    (RESULTS if results is None else results)[case["customer"]] = {
        "outcome": outcome, "message": case["message"], "reply": case["draft"], "model calls": case["calls"], "seconds": round(case["seconds"])}
    return case

def hand_to_human(case, why):
    print(f"\n{'#' * 72}\n TO A PERSON: {case['customer']} — {why}\n{'#' * 72}")
    print(f"kind: {case['kind']} · urgency: {case['urgency']} · triage said: {case['why']}\nfindings: {list(case['findings'])}"
          f"\nlast draft: {(case['draft'] or 'none')[:300]}\nreviewer said: {case['review']['reasons'] if case['review'] else '-'}")
    return finish(case, "to a person: " + why)

def close_after_triage(case):
    print(f"\n{'#' * 72}\n CLOSED after triage: {case['customer']} — nothing to look up, no reply drafted\n{'#' * 72}")
    return finish(case, "closed after triage")

def approve(case):
    print(f"\n{'#' * 72}\n APPROVED for {case['customer']} after {case['rounds']} round{'s' if case['rounds'] > 1 else ''}\n{'#' * 72}\n{case['draft']}")
    return finish(case, "approved")

def handle(message, max_rounds=2):
    """The supervisor: triage, dispatch, write, review, loop back, or hand over."""
    case = triage(new_case(message))
    if case["escalate"]:
        return hand_to_human(case, "escalated at triage")
    if not case["needs"]:                                                          # a message that needs nobody costs one call
        return close_after_triage(case)
    for name in case["needs"]:
        case = SPECIALISTS[name](case)
    reasons = None
    for round_no in range(1, max_rounds + 1):
        case["rounds"] = round_no
        case = reviewer(writer(case, reasons))
        if case["review"]["approved"]:
            return approve(case)
        reasons = case["review"]["reasons"]
        if round_no < max_rounds:                                                  # after the last round nobody would read the answer
            for name in case["review"]["ask_again"]:                              # the reviewer sends a specialist back to work
                case = SPECIALISTS[name](case, question=case["review"]["question"])
    return hand_to_human(case, f"the reviewer refused {max_rounds} times")

TRACE.clear()
case = handle(MESSAGES["tom"])

Read the trace top to bottom: who was called, in what order, what each one cost. In our test run Tom's case took two rounds and 8 calls. The reviewer returned the first draft for the same reasons as in Step 4, and the supervisor sent the account specialist back with those reasons as its question; the same row came back, since the account holds nothing about wording. The second draft said "a colleague will follow up to book a suitable slot" and was approved, with the £30 the `smart_meter` policy gives.

Now three more cases. Bayo's account shows four contacts this month, and the policy sends a third contact to the Complaints team, so watch triage take the exit before any specialist spends a call. Grace's thank-you needs nobody, so it should cost exactly one call.

In [ ]:
handle(MESSAGES["amina"])
handle(MESSAGES["bayo"])
handle(MESSAGES["thanks"])
pd.DataFrame(TRACE).groupby(["customer", "agent"], sort=False)[["model calls", "seconds"]].sum()

**What to look for.** In our test run Amina's first draft was returned by code alone ("immediately" is on the banned list), and the second was approved with her £86.40 balance and the policy's 10 working days: 7 calls. Bayo was escalated at triage by the model itself, thanks to the Exercise 1 prompt (the contacts rule would have caught him too): 1 call. Grace's thank-you was closed after 1 call. In the cost table, the writer and the reviewer take most of the seconds: every returned draft costs one more writer call and one more reviewer call.

## Is the team better than one agent?

The alternative to the team is **one agent with all the tools**: one prompt that gets the policy passages and the customer's account record at once and writes the reply, or hands the case to a person if the escalation rules say so. To make it a fair fight it gets the same retrieval (the two best passages), the same writing rules (`WRITER_RULES`, yours from Exercise 2), the **same reviewer** and the same budget of two rounds. What it does not get: triage, the code rules, and specialists that turn sources into short cited findings.

**Correct** is checked by code against the answer key below, read off the policy text and the customer table in section 2: Amina's reply must name her £86.40 credit balance, Tom's the £30 the `smart_meter` policy gives for a missed appointment (his balance of −£12.10 is money he owes), and neither may name any other amount; Bayo's case must be **escalated** to a person at once, not reach one after two refused drafts. This checks the amounts and the exit, not every sentence: read the replies too.

In [ ]:
ONE_AGENT_PROMPT = """You answer customer messages for Brightwave Energy, an energy supplier.

Customer message:
{message}

Our policy passages:
{passages}

The customer's account record (balance_gbp above 0 means the customer is in credit):
{account}

If the escalation policy says a person must take this case (a gas smell or safety risk, a legal threat, or a third or later contact about the same issue), reply only with "ESCALATE:" and the reason.
Otherwise write the reply to the customer.
Rules:
{rules}"""

ONE_RESULTS = {}

def one_agent(message, max_rounds=2):
    """The alternative to the team: one prompt sees the policy passages and the account record at once.
    Same retrieval, same writing rules, same reviewer, same two rounds; no triage, no code rules, no specialists."""
    case = new_case(message)
    hits = policies.query(query_texts=[message], n_results=2)
    case["findings"] = {"policy": "\n\n".join(f"[{m['source']}] {d}" for d, m in zip(hits["documents"][0], hits["metadatas"][0])),
                        "account": pd.read_sql("SELECT * FROM customers WHERE name = ?", conn, params=[case["customer"]]).to_string(index=False)}
    reasons = None
    for round_no in range(1, max_rounds + 1):
        started = time.time()
        reply = chat(ONE_AGENT_PROMPT.replace("{message}", message).replace("{passages}", case["findings"]["policy"])
                     .replace("{account}", case["findings"]["account"]).replace("{rules}", WRITER_RULES)
                     + (f"\n\nThe reviewer returned your previous draft:\n{case['draft']}\n\nbecause:\n" + "\n".join("- " + r for r in reasons)
                        + "\n\nFix every point." if reasons else ""), max_new_tokens=260)
        record(case, "one agent", reply, started, calls=1)
        if reply.strip().upper().startswith("ESCALATE"):
            return finish(case, "to a person: escalated by the agent", ONE_RESULTS)
        case["draft"], case["rounds"] = reply, round_no
        if reviewer(case)["review"]["approved"]:
            return finish(case, "approved", ONE_RESULTS)
        reasons = case["review"]["reasons"]
    return finish(case, f"to a person: the reviewer refused {max_rounds} times", ONE_RESULTS)

CORRECT = {                                       # the answer key, read off the policy text and the customer table in section 2
    "Amina Hassan": {"must": "86.4", "may": {"86.4", "250"}},   # refund_policy: her credit balance comes back (under the GBP 250 review limit)
    "Tom Okafor":   {"must": "30",   "may": {"30"}},            # smart_meter: GBP 30 for a missed appointment; -12.10 is his balance
    "Bayo Adeyemi": "escalated",                                  # escalation: a third contact about the same issue -> Complaints team, at once
}

def is_correct(customer, result):
    key = CORRECT[customer]
    if key == "escalated":
        return "escalated" in result["outcome"]                    # handed over at once, not after two refused drafts
    amounts = money(result["reply"] or "")
    return result["outcome"] == "approved" and key["must"] in amounts and amounts <= key["may"]

for customer in CORRECT:
    one_agent(next(m for m in MESSAGES.values() if customer in m))

rows = [{"path": path, "customer": customer, "outcome": r["outcome"], "correct": is_correct(customer, r),
         "model calls": r["model calls"], "seconds": r["seconds"]}
        for path, results in [("team", RESULTS), ("one agent", ONE_RESULTS)] for customer, r in results.items() if customer in CORRECT]
comparison = pd.DataFrame(rows)
print("3 cases: an illustration, not a measurement\n")
display(comparison)
comparison.groupby("path", sort=False).agg(approved=("outcome", lambda s: int((s == "approved").sum())), correct=("correct", "sum"),
                                          model_calls=("model calls", "sum"), seconds=("seconds", "sum"))

**What to look for.** In our test run both paths got two replies approved, and every approved reply named the right amount. The difference was Bayo. The team escalated him at triage, in one call. The single agent had the same escalation rule in its prompt and his four contacts in the account row, yet wrote a reply; the reviewer refused it twice, so he reached a person only after 4 calls, and the answer key counts that as wrong. Everywhere else the single agent was cheaper: 2 calls each for Amina and Tom against the team's 7 and 8, 8 calls in total against 16, and under half the seconds.

Read the approved replies as well, because the answer key checks only amounts: the single agent's reply to Tom ends with "We aim to resolve this fully within 10 working days", a promise no finding makes, and the reviewer let it through.

So, is the team better? Three cases cannot say, and that is the honest answer. Here the team bought one thing with twice the calls: a case like Bayo's goes to a person before anyone drafts a reply. That came from triage and a rule in code, and a code rule placed in front of a single agent would escalate Bayo the same way, by construction. Several of the team's extra calls went to drafts its own reviewer returned, so the critic is part of the price. A team pays off where its parts earn their calls: a check that must not fail, a specialist that turns a messy source into a short cited finding, a critic you have tested.

### Exercise 3 · A second critic, for tone only

The reviewer checks numbers in code and "does it answer the question, warmly, without unsupported claims" with the model: one checklist. Write `TONE_PROMPT` for a **second reviewer that judges tone only**: polite, warm, plain, no blame, no jargon, no curt or cold sentences, and explicitly *not* the facts or amounts. Keep `{message}` and `{draft}` where they are; the code adds the JSON format and runs your critic on every reply this lab produced, plus one deliberately curt test draft (written for this check) as a control: a tone critic that approves it is not checking tone.

Do the two critics agree? When they disagree, who wins? In a real system you decide that in code, for example: a reply goes out only when both approve.

In [ ]:
TONE_PROMPT = """You check the tone of a reply to a customer of an energy supplier. Judge the tone only: do not check facts, amounts or dates.
Approve the reply if it is polite, warm and plain: it acknowledges the customer's situation, does not blame the customer, uses no jargon, and has no curt, cold or dismissive sentences.
Reject it if any sentence is curt, cold, blaming or dismissive, and quote that sentence in your reasons.

Customer message:
{message}

Reply:
{draft}"""

def tone_reviewer(message, draft):
    """A second critic with a different checklist: tone only."""
    verdict = chat_json(TONE_PROMPT.replace("{message}", message).replace("{draft}", draft)
                        + '\n\nReturn JSON: {"approved": true/false, "reasons": ["each problem, if any"]}', max_new_tokens=150)
    return str(verdict.get("approved")).lower() == "true", verdict.get("reasons") or []

CURT = ("Your refund request has been noted. Refunds are paid within 10 working days. "
        "Please do not contact us again about this.\n\nBrightwave Customer Team")      # a test draft, written to be curt
drafts = [(path, customer, r["outcome"] == "approved", r["message"], r["reply"]) for path, results in [("team", RESULTS), ("one agent", ONE_RESULTS)]
          for customer, r in results.items() if r["reply"]]
drafts.append(("curt test draft", "Amina Hassan", "not run", MESSAGES["amina"], CURT))
rows = []
for path, customer, main_ok, message, draft in drafts:
    tone_ok, why = tone_reviewer(message, draft)
    rows.append({"draft": f"{path}: {customer}", "main reviewer approved": main_ok, "tone critic approved": tone_ok, "tone critic said": "; ".join(map(str, why))})
pd.set_option("display.max_colwidth", 300)                                      # show the critic's reasons in full
pd.DataFrame(rows)

**What to look for.** In our test run the tone critic rejected the curt test draft and quoted the right sentence ("Please do not contact us again about this."), so it does check tone. On the real replies the two critics disagreed twice: the tone critic rejected the team's approved reply to Tom, objecting to the sentence that begins "Per our policy, you are entitled...", and it approved the single agent's reply to Bayo that the main reviewer had refused twice. That is not a contradiction: they check different things. Whether a reply needs both approvals, or the tone critic only advises, is a decision for the supervisor's code, and it changes the cost (one more call per draft) and how often a case ends with a person.

## The same team as a graph — LangGraph

Everything the supervisor does is control flow over a shared state: run this, then that, branch on a verdict, loop with a budget, exit. That is exactly what a graph library exists to write down. **LangGraph** takes the case file as its state, the same agent functions as its nodes, and the supervisor's decisions as edges. Nothing about the model changes; the agents still call `chat()`.

What you gain is that the control flow becomes data: it can be drawn, checkpointed after every node, resumed after a human step, and handed to a colleague who has never read `handle()`. What you pay is a dependency and a layer of names. Build it by hand once, as you just did, and the framework stops being magic. The graph gets the one message nobody has handled yet: Sofia's.

In [ ]:
!pip install -q "langgraph>=0.6"

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class Case(TypedDict, total=False):                                          # the case file, declared as the graph's state
    message: str
    customer: str
    kind: str
    urgency: str
    needs: list
    escalate: bool
    why: str
    model_route: dict
    findings: dict
    draft: str
    review: dict
    rounds: int
    calls: int
    seconds: float

def specialists_node(case):
    """Run the specialists triage asked for, or the ones the reviewer sent back with a question."""
    sent_back = bool(case.get("review")) and bool(case["review"]["ask_again"])
    for name in (case["review"]["ask_again"] if sent_back else case["needs"]):
        case = SPECIALISTS[name](case, question=case["review"]["question"] if sent_back else None)
    return case

def writer_node(case):
    case["rounds"] = case.get("rounds", 0) + 1
    return writer(case, case["review"]["reasons"] if case.get("review") else None)

def after_triage(case):
    return "human" if case["escalate"] else "specialists" if case["needs"] else "closed"

def after_review(case, max_rounds=2):                                        # the supervisor's if-statements, as a routing function
    if case["review"]["approved"]:
        return "done"
    if case["rounds"] >= max_rounds:
        return "human"
    return "specialists" if case["review"]["ask_again"] else "writer"

graph = StateGraph(Case)
for name, fn in [("triage", triage), ("specialists", specialists_node), ("writer", writer_node), ("reviewer", reviewer),
                 ("done", approve), ("closed", close_after_triage),
                 ("human", lambda case: hand_to_human(case, "escalated at triage" if case["escalate"] else "the reviewer refused twice"))]:
    graph.add_node(name, fn)
graph.add_edge(START, "triage")
graph.add_conditional_edges("triage", after_triage, {"human": "human", "specialists": "specialists", "closed": "closed"})
graph.add_edge("specialists", "writer")
graph.add_edge("writer", "reviewer")
graph.add_conditional_edges("reviewer", after_review, {"done": "done", "human": "human", "writer": "writer", "specialists": "specialists"})
for last in ["done", "human", "closed"]:
    graph.add_edge(last, END)
team = graph.compile()

In [ ]:
from IPython.display import Image, display
try:
    display(Image(team.get_graph().draw_mermaid_png()))                        # drawn by the library from the edges above
except Exception:
    print(team.get_graph().draw_mermaid())                                     # the same diagram as text, if the renderer is unreachable

In [ ]:
TRACE.clear()
final = team.invoke(new_case(MESSAGES["sofia"]))                             # same agent functions, same case file, same trace
print(f"\n{final['customer']}: {final['calls']} model calls, {final['seconds']:.0f} s")

**What to look for.** The routing functions are `handle()`'s if-statements, so the graph takes the steps `handle()` would take: in our test run triage, both specialists, then writer and reviewer for two rounds, 7 calls, and Sofia's case went to a person after two refusals. Read the drafts against the findings. Sofia wrote that she is owed 62 pounds; the policy specialist, which sees her message, repeated "£62" as if it were a fact; the account says £61.50. The number check passed £62, because 62 appears in a finding. The second draft gave both figures, and only then did the reviewer's model check object to the difference. It also objected to "a colleague will follow up", which it had accepted for Tom. A critic built on a 4B model is not consistent from case to case; the trace is how you find out.

### Exercise 4 · Your own message

Write a customer message and send it through the team. Try a case the code rules do **not** cover, so that only your triage prompt can catch it: a gas smell (the escalation policy says: give the National Gas Emergency number and hand over to a person) or a legal threat. Or ask something the policies do not answer, and watch whether the writer admits it. Keep the `From:` line: it is how the case file knows the customer. Use a name from the customer table (Petra Novak, Lewis Campbell, ...) if you want the account specialist to find a record.

In [ ]:
MY_MESSAGE = """From: Petra Novak
Subject: Smell of gas

There has been a smell of gas near my meter cupboard since this morning. What should I do?"""

case = handle(MY_MESSAGE)

**What to look for.** In our test run the gas message was escalated at triage in one call, by the model: no code rule covers a gas smell, so the prompt did it. Notice what the hand-over does *not* do. The escalation policy also says to give the customer the National Gas Emergency number (0800 111 999), and this team sends nothing until a person picks up the case; a real system would send that number at once, from code. What did your message do? If triage got it wrong, would you fix it in the prompt, or is it a decision that belongs in code?

## Summary

| Agent | Decides | Reuses |
|---|---|---|
| Triage | kind, urgency, who is needed, escalate; two decisions backed by rules in code | the routing pattern from the Day 3 lecture |
| Policy specialist | what the rules allow, cited | the retrieval of the Day 2 RAG Pipeline lab |
| Account specialist | what the record shows | the SQL homework's query with repair |
| Writer | the reply, from findings only, under `WRITER_RULES` | |
| Reviewer | approve or return, with a specialist to ask again | code checks the numbers, the model checks the sense |
| Supervisor | order, budget, exit to a person, closing a case that needs nobody | the human approval gate of Day 3 Agent From Scratch, here as an exit |
| The supervisor, drawn | the same agent functions as nodes, the verdicts as edges | LangGraph `StateGraph` |

- A team is a **case file** plus small agents that each write their own part; the trace says who did what and what it cost.
- **Routing decides what the writer can know.** Without the policy finding, the writer turned Tom's −£12.10 balance into "£12.10 in compensation"; with it, the reply named the £30 the policy gives.
- **Decisions that must not fail belong in code**, next to the prompt: two rules backed up a first triage prompt that routed 1 of 5 messages as expected.
- **A critic is only as good as its checklist.** The number check cannot tell a balance from compensation; the model check returned correct drafts over wording and let an invented promise through. Test a critic on drafts you know are good and bad, like Exercise 3's curt draft.
- **Team or one agent?** On these three cases one agent got the same two replies approved for half the calls; the team got Bayo to a person at once. Three cases are an illustration, not a measurement.

---
# Extras (if you finish early)

**Optional code challenge: parallel specialists.** The policy and account specialists do not depend on each other. In LangGraph, two edges out of one node run both branches in the same step (a "fan-out"). Split `specialists` into a `policy` node and an `account` node, each with an edge from triage and an edge into the writer, and compare the `seconds` column with the run above. Two things to solve on the way: each node must return only the keys it changes, and a key that both branches write (`findings`, `calls`, `seconds`) needs a reducer, for example `Annotated[dict, merge_function]`. And before you predict a speed-up, think about what the two branches share: one model on one GPU.